In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model and tokenizer names
model_name = "Sci-fi-vy/Meditron-7b-finetuned" #Write your model names and just make changes as required

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto") #Use float16 for less memory usage and device_map="auto" to use GPU if available.

# Example prompt
prompt = "What are the symptoms of a common cold?"

# Tokenize the input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=200) # Adjust max_length as needed

# Decode the output
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print the response
print(response)

# Example with conditional generation, and temperature.
prompt2 = "A patient presents with fever, cough, and sore throat. What is the most likely diagnosis?"
inputs2 = tokenizer(prompt2, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs2 = model.generate(
        **inputs2,
        max_length=400,
        temperature=1, # Adjust temperature for creativity
        top_p=0.95,     # Adjust top_p for sampling
        top_k=50,       # Adjust top_k for sampling
        repetition_penalty=1.1, #Add repetition penalty to avoid repeating phrases.

    )

response2 = tokenizer.decode(outputs2[0], skip_special_tokens=True)
print(response2)

#Example of using a system prompt. This is useful for more guided responses.
system_prompt = "You are a medical expert. Answer medical questions accurately and concisely."
user_prompt = "What are the treatment options for hypertension?"

full_prompt = f"SYSTEM: {system_prompt}\nUSER: {user_prompt}\nASSISTANT:"

inputs3 = tokenizer(full_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs3 = model.generate(
        **inputs3,
        max_length=500,
        temperature=1,
        top_p=0.9,
        top_k=40,
        repetition_penalty=1.1
    )

response3 = tokenizer.decode(outputs3[0], skip_special_tokens=True)
print(response3)

tokenizer_config.json:   0%|          | 0.00/4.08k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/344 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from bert_score import score
from rouge import Rouge

# Load model and tokenizer
model_name = "Sci-fi-vy/Meditron-7b-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# Example prompts and expected answers
examples = [
    {
        "prompt": "What are the symptoms of a common cold?",
        "expected": "Common cold symptoms include sneezing, coughing, sore throat, and nasal congestion."
    },
    {
        "prompt": "A patient presents with fever, cough, and sore throat. What is the most likely diagnosis?",
        "expected": "The most likely diagnosis is a viral upper respiratory infection, such as the flu or common cold."
    },
    {
        "prompt": "What are the treatment options for hypertension?",
        "expected": "Treatment options include lifestyle changes, medications such as ACE inhibitors, beta-blockers, and diuretics."
    }
]

# Generate responses
responses = []
for example in examples:
    inputs = tokenizer(example["prompt"], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_length=200, temperature=0.7)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    responses.append(generated_text)

# Evaluation Metrics
rouge = Rouge()
bert_precision, bert_recall, bert_f1 = [], [], []
rouge_scores = []

for i, example in enumerate(examples):
    expected = example["expected"]
    generated = responses[i]

    # BERTScore
    P, R, F1 = score([generated], [expected], lang="en", model_type="microsoft/deberta-xlarge-mnli")
    bert_precision.append(P.mean().item() * 100)
    bert_recall.append(R.mean().item() * 100)
    bert_f1.append(F1.mean().item() * 100)

    # ROUGE Score
    rouge_score = rouge.get_scores(generated, expected)[0]["rouge-l"]["f"] * 100
    rouge_scores.append(rouge_score)

# Print Evaluation Results
print("\n==== Evaluation Results ====")
for i, example in enumerate(examples):
    print(f"\n📝 Prompt: {example['prompt']}")
    print(f"✅ Expected: {example['expected']}")
    print(f"🤖 Generated: {responses[i]}")
    print(f"📊 BERT F1 Score: {bert_f1[i]:.2f}%")
    print(f"📊 ROUGE-L Score: {rouge_scores[i]:.2f}%")

# Plot Accuracy Bar Graph
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
labels = ["Text", "Diagnosis", "Treatment"]
plt.bar(labels, bert_f1, color=["blue", "green", "orange"])
plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.title("Model Accuracy per Input Type")
for i, v in enumerate(bert_f1):
    plt.text(i, v - 5, f"{v:.2f}%", ha='center', fontsize=12, color='white', fontweight='bold')

plt.savefig("model_accuracy.jpeg", format="jpeg", dpi=400)
plt.show()

In [ ]:
!pip install rouge